# Delhi Logistics Protocol - Triad Causal Architecture
## Global Benchmark Notebook (METR-LA & PEMS-BAY)
This notebook strictly evaluates the TriadCausalSTGNN against the global 'Gold Standard' datasets.
It handles .h5 parsing, standard sliding-window sequencing, and computes standard MAE/RMSE/MAPE metrics.



In [ ]:
!pip install h5py pandas numpy torch networkx matplotlib seaborn
import os
import time
import json
import pickle
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment Setup Complete. PyTorch Version:", torch.__version__)



## 1. PEMS / METR-LA Data Loading Pipeline


In [ ]:
# Update these paths to match your Kaggle mounted dataset!
H5_FILE_PATH = "/kaggle/input/metr-la-dataset/metr-la.h5"
ADJ_FILE_PATH = "/kaggle/input/metr-la-dataset/adj_mx.pkl"

def load_graph_data(pkl_filename):
    print("Loading Adjacency Matrix...")
    try:
        with open(pkl_filename, 'rb') as f:
            sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')
        print(f"Graph loaded successfully! Shape: {adj_mx.shape}")
        return torch.tensor(adj_mx, dtype=torch.float32)
    except FileNotFoundError:
        print("WARNING: adj_mx.pkl not found! Falling back to Identity Matrix for testing.")
        # METR-LA has 207 nodes, PEMS-BAY has 325
        return torch.eye(207)

def load_h5_and_window(h5_filename, window_size=12, horizon=12, num_nodes=207):
    print(f"Loading H5 Data from {h5_filename}...")
    try:
        df = pd.read_hdf(h5_filename)
        data = df.values # (num_timesteps, num_nodes)
        
        # In real benchmarks, datasets often contain multiple features (e.g., speed, time of day). 
        # We'll expand it to shape (timesteps, nodes, 1 feature)
        data = np.expand_dims(data, axis=-1)
    except FileNotFoundError:
        print("WARNING: .h5 file not found! Generating mock data for compilation test.")
        # Generate 1 week of 5-minute interval mock data (2016 timesteps)
        data = np.random.rand(2016, num_nodes, 1)

    print(f"Raw data shape: {data.shape}")
    
    # 1. Normalize data (StandardScaler is typical for STGNN benchmarks)
    mean = np.mean(data)
    std = np.std(data)
    data = (data - mean) / (std + 1e-8)
    
    # 2. Sliding Window Generation
    print("Generating Spatiotemporal sequences...")
    x_offsets = np.arange(-window_size + 1, 1)
    y_offsets = np.arange(1, horizon + 1)
    
    num_samples = data.shape[0] - window_size - horizon + 1
    
    x_list = []
    y_list = []
    for i in range(num_samples):
        # x_window: (window_size, nodes, features)
        x_window = data[i + window_size - 1 + x_offsets]
        # y_window: (horizon, nodes, features)
        y_window = data[i + window_size - 1 + y_offsets]
        x_list.append(x_window)
        y_list.append(y_window)
        
    x = np.stack(x_list, axis=0) # (batch, window, nodes, features)
    y = np.stack(y_list, axis=0) # (batch, horizon, nodes, features)
    
    # Transpose to fit TriadCausalSTGNN: (batch, nodes, features, time)
    x = np.transpose(x, (0, 2, 3, 1))
    
    # For Y, typically we just predict the final step or average, but STGNN targets require (batch, nodes, out_channels)
    # We will use out_channels=horizon (12)
    y = np.squeeze(y, axis=-1) # (batch, horizon, nodes)
    y = np.transpose(y, (0, 2, 1)) # (batch, nodes, horizon)
    
    print(f"Final X shape: {x.shape}")
    print(f"Final Y shape: {y.shape}")
    
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), mean, std

# Load Graph
Adj = load_graph_data(ADJ_FILE_PATH)
num_nodes = Adj.shape[0]

# Load Sequences
# We truncate the dataset here for demonstration. In full training, use all samples.
X, Y, data_mean, data_std = load_h5_and_window(H5_FILE_PATH, num_nodes=num_nodes)
# Use first 500 samples for swift compilation and verification
X_train = X[:500]
Y_train = Y[:500]

# Synthesize Confounders (Z)
# Real benchmarks usually don't provide explicit "Festival" labels in METR-LA, 
# so we extract basic temporal confounders (like Day/Night cycle) implicitly from index.
Z_train = torch.ones((X_train.shape[0], 2)) # Dummy Confounders



## 2. Triad Causal STGNN Architecture


In [ ]:
class Layer1_DataDriven(nn.Module):
    def __init__(self, num_nodes, embed_dim, in_channels, out_channels):
        super().__init__()
        self.node_embed = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        # Deep Dilated TCN for larger temporal receptive field
        self.tcn1 = nn.Conv1d(in_channels, 32, kernel_size=3, padding=1, dilation=1)
        self.tcn2 = nn.Conv1d(32, out_channels, kernel_size=3, padding=2, dilation=2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        adaptive_adj = torch.softmax(torch.matmul(self.node_embed, self.node_embed.T), dim=-1)
        batch, nodes, features, time = x.shape
        x_reshaped = x.reshape(batch * nodes, features, time)
        tcn_out = self.relu(self.tcn1(x_reshaped))
        tcn_out = self.tcn2(tcn_out)
        tcn_out = tcn_out.reshape(batch, nodes, -1, time)
        return adaptive_adj, tcn_out

class TrafficPhysicsPrior(nn.Module):
    def __init__(self, physical_adj):
        super().__init__()
        self.physical_adj = nn.Parameter(physical_adj, requires_grad=False)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
    def forward(self, adaptive_adj):
        return self.alpha * self.physical_adj + (1 - self.alpha) * adaptive_adj

class CausalDeconfounder(nn.Module):
    def __init__(self, num_nodes, num_confounders):
        super().__init__()
        self.confounder_weight = nn.Parameter(torch.randn(num_confounders, num_nodes, num_nodes) * 0.1)
        
    def forward(self, hybrid_adj, z):
        batch_size = hybrid_adj.shape[0]
        causal_correction = torch.einsum('bc,cnm->bnm', z, self.confounder_weight)
        # Fix dying ReLU: Use Sigmoid gating instead of subtraction
        gate = torch.sigmoid(causal_correction)
        causal_adj = hybrid_adj * gate
        return causal_adj

class TriadCausalSTGNN(nn.Module):
    def __init__(self, num_nodes, embed_dim, in_channels, out_channels, num_confounders, physical_adj):
        super().__init__()
        self.layer1 = Layer1_DataDriven(num_nodes, embed_dim, in_channels, out_channels)
        self.layer2 = TrafficPhysicsPrior(physical_adj)
        self.layer3 = CausalDeconfounder(num_nodes, num_confounders)
        # Replace the gn_weight with a proper output projection layer 
        # (out_channels * time_steps) -> forecasting_horizon
        self.output_proj = nn.Linear(out_channels * 12, 12)

    def forward(self, x, z):
        adaptive_adj, tcn_features = self.layer1(x)
        hybrid_adj = self.layer2(adaptive_adj)
        hybrid_adj_batch = hybrid_adj.unsqueeze(0).expand(x.shape[0], -1, -1)
        causal_adj = self.layer3(hybrid_adj_batch, z)
        
        # Spatiotemporal Fusion (Do not slice the final time step!)
        batch, nodes, channels, time = tcn_features.shape
        tcn_flat = tcn_features.reshape(batch, nodes, channels * time)
        
        # Spatial Graph Convolution over the entire temporal context
        causal_embeds = torch.bmm(causal_adj, tcn_flat)
        
        # Final projection to match the Y horizon target shape (batch, nodes, horizon)
        causal_embeds = self.output_proj(causal_embeds)
        return causal_embeds, causal_adj

# Initialize standard benchmark parameters
# in_channels = 1 (speed), out_channels = 12 (predicting next 12 time steps/1 hour)
model = TriadCausalSTGNN(
    num_nodes=num_nodes, 
    embed_dim=32, 
    in_channels=1, 
    out_channels=12, 
    num_confounders=2, 
    physical_adj=Adj
)



## 3. Official Benchmarking Loop


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# MAE and MAPE Metrics
def masked_mae(preds, labels, null_val=0.0):
    mask = labels != null_val
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels)
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)

def masked_mape(preds, labels, null_val=0.0):
    mask = labels != null_val
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels) / (labels + 1e-8)
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)

optimizer = optim.Adam(model.parameters(), lr=0.005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
criterion = nn.MSELoss()

# Create DataLoader to prevent Kaggle OOM
dataset = TensorDataset(X_train, Y_train, Z_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

epochs = 100
history_loss = []
history_mae = []

print("Starting Benchmark Training...")
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    epoch_mae = 0.0
    
    for batch_x, batch_y, batch_z in dataloader:
        optimizer.zero_grad()
        
        preds, causal_adj = model(batch_x, batch_z)
        
        # We train on MSE (normalized)
        loss = criterion(preds, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Calculate MAE for logging
        preds_orig = preds.detach() * data_std + data_mean
        y_train_orig = batch_y * data_std + data_mean
        epoch_mae += masked_mae(preds_orig, y_train_orig).item()
        
    scheduler.step()
    
    avg_loss = epoch_loss / len(dataloader)
    avg_mae = epoch_mae / len(dataloader)
    
    history_loss.append(avg_loss)
    history_mae.append(avg_mae)
    
    if epoch % 10 == 0:
        rmse = torch.sqrt(torch.mean((preds_orig - y_train_orig)**2)).item()
        mape = masked_mape(preds_orig, y_train_orig).item()
        print(f"Epoch {epoch:03d}/{epochs} | Loss: {avg_loss:.4f} | MAE: {avg_mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape*100:.2f}%")

print("Benchmark Training Complete!")
torch.save(model.state_dict(), "causal_stgnn_benchmark_weights.pth")



## 4. Benchmark Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Convergence Curve
axes[0].plot(history_loss, label="Training Loss (MSE)", color="#3b82f6", linewidth=2)
axes[0].set_title("Benchmark Convergence")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Normalized Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# 2. Causal Attention Heatmap
final_adj = causal_adj[0].detach().numpy()
sns.heatmap(final_adj[:20, :20], cmap="viridis", ax=axes[1], cbar=True)
axes[1].set_title("Learned Causal Graph (First 20 Nodes)")
axes[1].set_xlabel("Target Node")
axes[1].set_ylabel("Source Node")

plt.tight_layout()
plt.show()

